# Mouse exon and intron smiFISH probe sets

Designs separate **exon** and **intron** sets for Spen, Arid5b, Jarid2, Sfmbt2, Sfmbt1 and Mdm4 with the integrated Python Oligostan port. Each set targets 40 selected probes, with a minimum of 30 when enough candidates pass. The selected transcript is the longest spliced RefGene transcript in mm10; intron intervals exclude exons from all annotated isoforms of that gene. Probe oligos are reverse complements of the target RNA. Coordinates are mm10/GRCm38, 1-based inclusive.

All selected probes must have an exact full-length BLAST hit at the intended genomic position and no other BLAST alignment covering at least 80% of the probe at at least 90% identity. Review the audit tables and the generated order sheet before synthesis. The full mouse Spen comparison to the original R script passed under the matched fixed −32, masking-off settings used here. Automatic dG optimization and RepeatMasker remain separate parity gates; see `oligostan/PARITY_PLAN.md`.

Run top to bottom. Create an ignored `.smifish-local.json` beside `oligostan/` using `oligostan/mouse_smifish.example.json`. Point it to the mm10 FASTA, `.fai`, GTF, BLAST database, `blastn`, and output directory. Progress bars appear during annotation loading, candidate generation, BLAST and export.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

workdir = Path.cwd()
package_parent = next((p for p in (workdir, workdir / "probe_and_primer_design", workdir.parent / "probe_and_primer_design") if (p / "oligostan" / "__init__.py").is_file()), None)
if package_parent is None:
    raise FileNotFoundError("Run this notebook from the repository or probe_and_primer_design directory")
sys.path.insert(0, str(package_parent))

from oligostan.mouse_smifish import (
    GENES, load_run_config, load_models, generate_candidates,
    adaptive_blast_verify, select_sets, write_outputs,
)

config = load_run_config()
print("Reference: mm10/GRCm38; targets:", ", ".join(GENES))
print("Output parent:", config["output_parent"])


## 1. Load transcript annotations

The longest spliced RefGene transcript is chosen for each gene. Exon probes remain within a single exon. Intronic regions are stripped of any exon from another annotated isoform.


In [ ]:
models = load_models(config["gtf"])
display(__import__("pandas").DataFrame([
    {"gene": gene, "transcript": model["transcript_id"], "chrom": model["chrom"],
     "strand": model["strand"], "exons": len(model["exon"]),
     "eligible_introns": len(model["intron"])}
    for gene, model in models.items()
]))


## 2. Design Oligostan candidates

The core algorithm optimizes 26–32 nt probe lengths near dG37 = −32. The package's GC 0.4–0.6 and PNAS 1/2/4 settings are the baseline; additional candidates are assigned documented relaxation tiers. Long introns are tiled and tiles are interleaved across introns. `intron_pool` is the approximate per-gene ceiling for filtered intron candidates before BLAST; increase it if any intron set is short.


In [ ]:
candidates = generate_candidates(models, fasta=config["fasta"], intron_pool=600)
display(candidates.groupby(["gene", "region"]).size().rename("filtered_candidates").reset_index())


## 3. Verify every candidate with local BLAST

`blastn-short` searches both strands of the mm10 genome with dust and soft masking disabled. The baseline tier is searched first. If any set has fewer than 30 verified probes, the next quality tier is BLAST checked for that set. A probe passes only with an exact full-length intended hit and no strong off-target. The hit audit, including non-disqualifying weak matches returned by BLAST, is exported.


In [ ]:
verified, blast_hits, blast_command = adaptive_blast_verify(candidates, blastn=config["blastn"], database=config["blast_db"], threads=8)
display(verified.groupby(["gene", "region"])["blast_verified"].agg(["sum", "count"]).reset_index())


## 4. Assemble sets and inspect shortfalls

Selected probes are spread across the available target span while retaining high Oligostan scores. A set with fewer than 30 verified probes is explicitly flagged in the summary; it is never padded with unverified oligos. The `quality_tier` column shows how far filters were relaxed for each probe.


In [ ]:
selected, summary = select_sets(verified, minimum=30, target_count=40)
display(summary)
if not summary["meets_minimum"].all():
    print("Some sets have fewer than 30 BLAST-verified probes; inspect set_summary.csv and consider increasing intron_pool or revisiting filter settings.")
else:
    print("All 12 sets have at least 30 BLAST-verified probes.")


## 5. Save the complete design and BLAST audit

A fresh subfolder is created under the configured `output_parent` directory. Each gene/region folder contains verified probe CSV, FASTA and a Flap X order sheet marked for review. The root includes all designed candidates, BLAST-tested candidates, BLAST hits, selected probes, manifest and set summary.


In [ ]:
output_dir = write_outputs(models, candidates, verified, blast_hits, selected, summary, blast_command, output_parent=config["output_parent"], reference_paths=config)
print("Saved:", output_dir)
display(summary[["gene", "region", "selected_count", "meets_minimum"]])
